In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import Libraries 

In [2]:
# Libraries for data obtaining
from pathlib import Path
# Libraries for model architecture
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
#Download processed dataset from Zenodo
!wget -O /kaggle/working/data.tar.gz "https://zenodo.org/records/2652278/files/data.tar.gz?download=1"

--2026-04-27 11:32:24--  https://zenodo.org/records/2652278/files/data.tar.gz?download=1
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.184.98.114, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 539905064 (515M) [application/octet-stream]
Saving to: ‘/kaggle/working/data.tar.gz’

g/data.tar.gz        52%[=========>          ] 270.50M  1.06MB/s    eta 4m 23s 

In [ ]:
#Extract data
!tar -xzf /kaggle/working/data.tar.gz -C /kaggle/working/

In [ ]:
root = Path("/kaggle/working/data")
 
# Look for directories that contain train.csv/valid.csv/test.csv
all_dirs = sorted([d for d in root.rglob("*") if d.is_dir()])
cell_type_dirs = sorted([d for d in all_dirs if (d / "train.csv").exists()])
print(f"  Cell type directories found: {len(cell_type_dirs)}")

In [ ]:
#Inspecting one cell type in detail
e047_dir = next((d for d in cell_type_dirs if d.name == "E047"), cell_type_dirs[0])
print(f"  Path: {e047_dir}")
 
splits = {}
for split in ["train", "valid", "test"]:
    fpath = e047_dir / f"{split}.csv"
    df = pd.read_csv(fpath, header=None)
    splits[split] = df
    print(f"\n  {split}.csv:")
    print(f"    Shape:   {df.shape}")
    print(f"    Columns: {df.shape[1]} total")
    print(f"    First row (first 10 values): {df.iloc[0, :10].tolist()}")
    print(f"    Last column (label?): unique values = {sorted(df.iloc[:, -1].unique())}")
    print(f"    Second-to-last col sample: {df.iloc[:3, -2].tolist()}")
 

In [ ]:
splits['train'].head()

##### Dataset columns are:
- GeneID
- Bin ID
- H3K27me3 count
- H3K36me3 count
- H3K4me1 count
- H3K4me3 count
- H3K9me3 counts
- Binary Label for gene expression (0/1)
